This colab file
1. Sets up the Environment by importing appropriate Libraries
2. Uses a prepared dataset of female patients suffering from Blood diseases who were admitted to ICU and were administered various medications
3. Cleanses the dataset by removing duplicates and normalizing medicine names
4. Recommends medication based on lower mortality risk
5. Feature Engineering to convert medication list to binary columns.
6. Uses Random Forest Classifier as prediction model
7. Recommends medications that lowers predicted risk.
8. Prints the results.
9. Save the trained model.
10. Download the trained model



In [1]:
# ============================================================
# SECTION 1: INSTALL & IMPORT LIBRARIES
# ============================================================
# These libraries are required for:
# - Data handling (pandas, numpy)
# - Machine learning (sklearn)
# - Visualization (matplotlib)
# ============================================================

!pip install pandas numpy scikit-learn matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import numpy as np


In [2]:
# ============================================================
# SECTION 2: LOAD DATASET
# ============================================================
# Load your dataset containing:
# - ICU female patients
# - Blood disease diagnosis
# - Medications prescribed
#
# Expected file:
# female_icu_blood_disease_with_medications.csv
# ============================================================

from google.colab import files

# Upload file manually
uploaded = files.upload()

# Read CSV
df = pd.read_csv("female_icu_blood_disease_with_medications.csv")

# Preview dataset
print("Dataset Shape:", df.shape)
df.head()

Saving female_icu_blood_disease_with_medications.csv to female_icu_blood_disease_with_medications.csv
Dataset Shape: (633247, 15)


,subject_id,hadm_id,icustay_id,gender,admittime,dischtime,deathtime,hospital_expire_flag,mortality,intime,outtime,icd9_code,drug,drug_name_generic,route
0,6,107064,228232,F,2175-05-30 07:15:00,2175-06-15 16:00:00,NaN,0,0,2175-05-30 21:30:54,2175-06-03 13:39:54,2859,1/2 ns,NaN,IV
1,6,107064,228232,F,2175-05-30 07:15:00,2175-06-15 16:00:00,NaN,0,0,2175-05-30 21:30:54,2175-06-03 13:39:54,2859,acetaminophen,acetaminophen,PO
2,6,107064,228232,F,2175-05-30 07:15:00,2175-06-15 16:00:00,NaN,0,0,2175-05-30 21:30:54,2175-06-03 13:39:54,2859,albuterol,albuterol inhaler,IH
3,6,107064,228232,F,2175-05-30 07:15:00,2175-06-15 16:00:00,NaN,0,0,2175-05-30 21:30:54,2175-06-03 13:39:54,2859,amlodipine,amlodipine,PO
4,6,107064,228232,F,2175-05-30 07:15:00,2175-06-15 16:00:00,NaN,0,0,2175-05-30 21:30:54,2175-06-03 13:39:54,2859,anti-thymocyte globulin (rabbit),NaN,IV


In [3]:
# ============================================================
# SECTION 3: DATA CLEANING
# ============================================================
# Goals:
# 1. Remove missing values
# 2. Normalize medication names
# 3. Remove duplicates
# 4. Prepare structured medication lists per patient
# ============================================================

# Drop rows with missing critical fields
df = df.dropna(subset=['subject_id', 'hadm_id', 'drug'])

# Normalize drug names (lowercase for consistency)
df['drug'] = df['drug'].str.lower().str.strip()

# Remove duplicate rows
df = df.drop_duplicates()

print("After cleaning:", df.shape)

After cleaning: (633247, 15)


In [4]:
# ============================================================
# SECTION 4: MEDICATION RECOMMENDATION BASED ON LOWER MORTALITY RISK
# ============================================================
# Goal:
# Recommend medications associated with LOWER predicted mortality risk.
#
# IMPORTANT:
# This requires a dataset with BOTH:
#   mortality = 0  survived
#   mortality = 1  died
#
# NOTE: If CSV only contains survivors, this section cannot estimate
# mortality reduction correctly.
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Check required mortality column
# ------------------------------------------------------------

if "mortality" not in df.columns:
    raise ValueError("""
    Your dataset does not contain a 'mortality' column.

    To recommend medications based on lowering mortality risk,
    the dataset must include both survivors and non-survivors.

    Required column:
      mortality = 0 for survived
      mortality = 1 for died
    """)

# ------------------------------------------------------------
# Clean mortality label
# ------------------------------------------------------------

df["mortality"] = pd.to_numeric(df["mortality"], errors="coerce")
df = df.dropna(subset=["mortality"])
df["mortality"] = df["mortality"].astype(int)

print(df["mortality"].value_counts())

# ------------------------------------------------------------
# Create one row per admission
# ------------------------------------------------------------

patient_level = (
    df.groupby("hadm_id")
      .agg({
          "subject_id": "first",
          "gender": "first",
          "icd9_code": lambda x: list(set(x.dropna())),
          "drug": lambda x: list(set(x.dropna().astype(str).str.lower().str.strip())),
          "mortality": "max"
      })
      .reset_index()
)

patient_level.head()

mortality
0    518239
1    115008
Name: count, dtype: int64


,hadm_id,subject_id,gender,icd9_code,drug,mortality
0,100028,53456,F,[2875],"[famotidine, calcium gluconate, ondansetron, d...",0
1,100036,30078,F,[28521],"[furosemide, calcium gluconate, enoxaparin sod...",0
2,100038,21234,F,[2859],"[furosemide, docusate sodium, aluminum-magnesi...",0
3,100045,1569,F,"[28521, 2875]","[lidocaine 0.5%/epinephrine, rifaximin, ampici...",0
4,100060,6828,F,[2859],"[acetaminophen, ns, docusate sodium, quetiapin...",0


In [5]:
# ============================================================
# SECTION 5: MEDICATION FEATURE ENGINEERING
# ============================================================
# Convert medication lists into binary columns.
#
# Example:
#   aspirin = 1 if patient received aspirin
#   aspirin = 0 otherwise
# ============================================================

mlb = MultiLabelBinarizer()

med_features = mlb.fit_transform(patient_level["drug"])

med_feature_df = pd.DataFrame(
    med_features,
    columns=[f"med_{m}" for m in mlb.classes_]
)

model_df = pd.concat(
    [patient_level[["hadm_id", "mortality"]], med_feature_df],
    axis=1
)

print("Model dataset shape:", model_df.shape)
model_df.head()

Model dataset shape: (8504, 2201)


,hadm_id,mortality,med_*ind* pexelizumab/placebo,med_*nf,med_*nf* allopurinol sodium,med_*nf* arginine hcl,med_*nf* basiliximab,med_*nf* beclomethasone dipropionate inhalation,med_*nf* benzoyl peroxide 5% wash,med_*nf* budesonide,...,med_ziprasidone mesylate,med_zithromax z-pak,med_zofran,med_zoledronic acid,med_zolpidem tartrate,med_zonisamide,med_zosyn desensitization,med_zyflo,med_zymar,med_zyrtec
0,100028,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,100036,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,100038,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,100045,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,100060,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
# ============================================================
# SECTION 6: TRAIN MORTALITY PREDICTION MODEL
# ============================================================
# Model:
# Random Forest Classifier
#
# Target:
# mortality
#
# Interpretation:
# Medications that reduce predicted probability of mortality
# may be considered candidate beneficial medications.
# ============================================================

X = model_df.drop(columns=["hadm_id", "mortality"])
y = model_df["mortality"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

mortality_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced"
)

mortality_model.fit(X_train, y_train)

# Predict mortality risk
y_prob = mortality_model.predict_proba(X_test)[:, 1]

auroc = roc_auc_score(y_test, y_prob)
auprc = average_precision_score(y_test, y_prob)

print("AUROC:", round(auroc, 3))
print("AUPRC:", round(auprc, 3))

AUROC: 0.913
AUPRC: 0.628


In [7]:
# ============================================================
# SECTION 7: RECOMMEND MEDICATIONS THAT LOWER PREDICTED RISK
# ============================================================
# For a selected patient:
# 1. Calculate current predicted mortality risk
# 2. Simulate adding each candidate medication
# 3. Recalculate predicted risk
# 4. Rank medications by largest risk reduction
# ============================================================

def recommend_meds_lower_mortality(patient_index, top_n=10):
    patient_vector = X.iloc[[patient_index]].copy()

    current_risk = mortality_model.predict_proba(patient_vector)[0, 1]

    recommendations = []

    for med_col in X.columns:
        # Skip medications already present
        if patient_vector.iloc[0][med_col] == 1:
            continue

        simulated_patient = patient_vector.copy()
        simulated_patient[med_col] = 1

        new_risk = mortality_model.predict_proba(simulated_patient)[0, 1]

        risk_reduction = current_risk - new_risk

        recommendations.append({
            "medication": med_col.replace("med_", ""),
            "current_risk": current_risk,
            "predicted_risk_if_added": new_risk,
            "risk_reduction": risk_reduction
        })

    rec_df = pd.DataFrame(recommendations)

    rec_df = rec_df.sort_values(
        by="risk_reduction",
        ascending=False
    )

    return rec_df.head(top_n)

# Example patient
patient_index = 0

recommendations = recommend_meds_lower_mortality(
    patient_index=patient_index,
    top_n=10
)

recommendations

,medication,current_risk,predicted_risk_if_added,risk_reduction
1541,oxycodone-acetaminophen,0.289207,0.256864,0.032343
1178,lisinopril,0.289207,0.266367,0.022840
1087,ketorolac,0.289207,0.266420,0.022787
1443,neostigmine,0.289207,0.270520,0.018687
2127,warfarin,0.289207,0.277891,0.011316
2158,zolpidem tartrate,0.289207,0.278606,0.010601
1212,magnesium oxide,0.289207,0.279794,0.009413
438,cepacol (menthol),0.289207,0.280867,0.008340
905,golytely,0.289207,0.281380,0.007827
192,amlodipine,0.289207,0.281520,0.007687


In [8]:
# ============================================================
# SECTION 8: PRINT CLINICAL DECISION-SUPPORT RESULTS
# ============================================================

print("Patient Admission ID:")
print(patient_level.iloc[patient_index]["hadm_id"])

print("\nExisting medications:")
print(patient_level.iloc[patient_index]["drug"])

print("\nTop medications associated with lower predicted mortality risk:")
display(recommendations)

print("""
Clinical Interpretation:
These medications are NOT automatic treatment orders.

They are candidate medications associated with lower model-predicted
mortality risk based on historical patient patterns.

Thorough Doctor Review is mandatory.
""")

Patient Admission ID:
100028

Existing medications:
['famotidine', 'calcium gluconate', 'ondansetron', 'docusate sodium', 'insulin', 'neutra-phos', 'lidocaine 1%', 'meperidine', '0.9% sodium chloride (mini bag plus)', 'promethazine hcl', 'haloperidol', 'iso-osmotic sodium chloride', '0.9% sodium chloride', 'ampicillin-sulbactam', 'oxycodone (immediate release)', 'potassium chloride', 'acetaminophen', 'milk of magnesia', 'sodium chloride 0.9%  flush', 'metoprolol tartrate', 'dextrose 50%', 'lr', 'potassium chl 20 meq / 1000 ml d5 1/2 ns', 'bisacodyl', 'hydromorphone (dilaudid)', 'glucagon', 'potassium phosphate', '5% dextrose', 'heparin', 'bag', 'influenza virus vaccine', 'd5 1/2ns', 'diltiazem', 'magnesium sulfate', 'pneumococcal vac polyvalent']

Top medications associated with lower predicted mortality risk:


,medication,current_risk,predicted_risk_if_added,risk_reduction
1541,oxycodone-acetaminophen,0.289207,0.256864,0.032343
1178,lisinopril,0.289207,0.266367,0.022840
1087,ketorolac,0.289207,0.266420,0.022787
1443,neostigmine,0.289207,0.270520,0.018687
2127,warfarin,0.289207,0.277891,0.011316
2158,zolpidem tartrate,0.289207,0.278606,0.010601
1212,magnesium oxide,0.289207,0.279794,0.009413
438,cepacol (menthol),0.289207,0.280867,0.008340
905,golytely,0.289207,0.281380,0.007827
192,amlodipine,0.289207,0.281520,0.007687



Clinical Interpretation:
These medications are NOT automatic treatment orders.

They are candidate medications associated with lower model-predicted
mortality risk based on historical patient patterns.

Thorough Doctor Review is mandatory.



In [11]:
# ============================================================
# SAVE TRAINED MODEL
# ============================================================

import joblib

# Save model
joblib.dump(mortality_model, "female_blood_patients_meds_Recmd_model.pkl")

# (Optional) Save scaler if used
# joblib.dump(scaler, "scaler.pkl")

print("Model saved successfully")

Model saved successfully


In [12]:
# ============================================================
# DOWNLOAD TRAINED MODEL
# ============================================================
from google.colab import files
files.download("female_blood_patients_meds_Recmd_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>